#Games after Trade Deadline (Aug 4th - Sept 27)

#Record Before End of Trade Deadline (March 25 - Aug 3)

In [0]:
# %sql
# -- Total runs per team
# CREATE OR REPLACE TEMP VIEW team_game_runs AS
# SELECT
#     game_pk,
#     game_date,
#     team_id,
#     SUM(runs) AS runs_scored
# FROM gold.fact_batting_stats
# GROUP BY game_pk, game_date, team_id
# HAVING game_date <= "2026-08-03";



# -- Total runs scored per game
# CREATE OR REPLACE TEMP VIEW game_results AS
# SELECT 
#     g.game_pk,
#     g.game_date,
#     g.home_team_id,
#     h.runs_scored AS home_runs,
#     g.away_team_id,
#     a.runs_scored AS away_runs
# FROM gold.dim_games AS g
# LEFT JOIN team_game_runs AS h ON g.game_pk = h.game_pk AND g.home_team_id = h.team_id
# LEFT JOIN team_game_runs AS a ON g.game_pk = a.game_pk AND g.away_team_id = a.team_id
# WHERE g.game_status IN ('Final', 'Pre-Game', 'Scheduled', 'Completed Early');



# -- wins and losses per team per game
# CREATE OR REPLACE TEMP VIEW team_outcomes AS
# SELECT
#     game_pk,
#     game_date,
#     home_team_id AS team_id,
#     CASE WHEN home_runs > away_runs THEN 1 ELSE 0 END AS Win,
#     CASE WHEN home_runs < away_runs THEN 1 ELSE 0 END AS Loss,
#     home_runs - away_runs AS run_diff
# FROM game_results
# UNION ALL
# SELECT
#     game_pk,
#     game_date,
#     away_team_id AS team_id,
#     CASE WHEN away_runs > home_runs THEN 1 ELSE 0 END AS Win,
#     CASE WHEN away_runs < home_runs THEN 1 ELSE 0 END AS Loss,
#     away_runs - home_runs AS run_diff
# FROM game_results;


# -- Overall record
# CREATE OR REPLACE TEMP VIEW team_record AS
# SELECT
#     t.team_name,
#     SUM(Win) AS Wins,
#     SUM(Loss) AS Losses
# FROM team_outcomes AS o
# LEFT JOIN gold.dim_teams AS t
# ON o.team_id = t.team_id
# GROUP BY t.team_name

In [0]:
# %sql
# SELECT * FROM team_record ORDER BY Wins DESC

In [0]:
# manual_corrections = [
#     {
#         "team_name": "Philadelphia Phillies",
#         "wins_adjustment": 1,
#         "losses_adjustment": 0
#     },
#     {
#         "team_name": "Washington Nationals",
#         "wins_adjustment": 0,
#         "losses_adjustment": 1
#     },
#     {
#         "team_name": "St. Louis Cardinals",
#         "wins_adjustment": 1,
#         "losses_adjustment": 0
#     },
#     {
#         "team_name": "New York Yankees",
#         "wins_adjustment": 0,
#         "losses_adjustment": 1
#     },
#     {
#         "team_name": "Pittsburgh Pirates",
#         "wins_adjustment": 1,
#         "losses_adjustment": 0
#     },
#     {
#         "team_name": "Milwaukee Brewers",
#         "wins_adjustment": 0,
#         "losses_adjustment": 1
#     },
#     {
#         "team_name": "San Francisco Giants",
#         "wins_adjustment": 1,
#         "losses_adjustment": 0
#     },
#     {
#         "team_name": "Texas Rangers",
#         "wins_adjustment": 0,
#         "losses_adjustment": 1
#     },
#     {
#         "team_name": "Los Angeles Dodgers",
#         "wins_adjustment": 0,
#         "losses_adjustment": 1
#     },
#     {
#         "team_name": "Toronto Blue Jays",
#         "wins_adjustment": 1,
#         "losses_adjustment": 0
#     }
# ]

In [0]:
# deadline_corrections_df = spark.createDataFrame(manual_corrections)

# deadline_corrections_df.write.mode("overwrite").saveAsTable("gold.mlb_manual_deadline_record_corrections")

In [0]:
# %sql
# CREATE OR REPLACE TABLE gold.mlb_deadline_standings AS

# SELECT

#     s.team_name,

#     s.Wins
#         + COALESCE(c.wins_adjustment, 0)
#         AS Wins,

#     s.Losses
#         + COALESCE(c.losses_adjustment, 0)
#         AS Losses,

#     ROUND((s.Wins + COALESCE(c.wins_adjustment, 0)) / ((s.Wins + COALESCE(c.wins_adjustment, 0)) + (s.Losses + COALESCE(c.losses_adjustment, 0))), 4) AS win_pct

# FROM team_record s

# LEFT JOIN gold.mlb_manual_deadline_record_corrections c
#     ON s.team_name = c.team_name;

In [0]:
# %sql
# SELECT * FROM gold.mlb_deadline_standings

#Future Games After Trade Deadline (Aug 4th - Sept 27)

In [0]:
# %sql

# CREATE OR REPLACE TEMP VIEW future_games AS

# -- Grabbing the future game details from the bronze layer's JSON
# WITH exploded_games AS (

#     SELECT
#         game.gamePk AS game_pk,

#         CAST(
#             game.officialDate
#             AS DATE
#         ) AS game_date,

#         game.gameType AS game_type,
#         game.teams.away.team.id AS away_team_id,
#         game.teams.home.team.id AS home_team_id,

#         game.status.detailedState AS game_status,

#         s.ingestion_timestamp

#     FROM bronze.mlb_schedule_raw s

#     LATERAL VIEW EXPLODE(
#         FROM_JSON(
#             GET_JSON_OBJECT(
#                 s.raw_json,
#                 '$.dates[0].games'
#             ),

#             'array<struct<
#                 gamePk:bigint,
#                 gameDate:string,
#                 officialDate:string,
#                 gameType:string,
#                 status:struct<detailedState:string>,
#                 teams:struct<
#                     away:struct<
#                         team:struct<id:bigint>
#                     >,
#                     home:struct<
#                         team:struct<id:bigint>
#                     >
#                 >
#             >>'
#         )
#     ) exploded AS game

#     -- Regualar season only
#     WHERE game.gameType = 'R'

#         AND game.status.detailedState IN (
#             'Scheduled',
#             'Pre-Game',
#             'Final'
#         )

#         --Ignore old schedule snapshots
#         AND CAST(game.officialDate AS DATE) >= "2026-08-04"
# ),

# -- finding duplicate future games (if any)
# deduplicated AS (
#     SELECT
#         *,
#         ROW_NUMBER() OVER (
#             PARTITION BY game_date, home_team_id, away_team_id
#             ORDER BY ingestion_timestamp DESC
#         ) AS rn
#     FROM exploded_games
# )

# -- Game details from Aug 4th - Sept 27
# -- matching team_ids with team name
# SELECT
#     f.game_pk,
#     f.game_date,
#     f.game_type,
#     f.away_team_id,
#     away.team_name AS away_team,
#     home_team_id,
#     home.team_name AS home_team,
#     f.game_status

# FROM deduplicated f
# LEFT JOIN silver.mlb_teams away
# ON f.away_team_id = away.team_id
# LEFT JOIN silver.mlb_teams home
# ON f.home_team_id = home.team_id

# WHERE rn = 1;

#10,000 Simulations with win % at the Trade Deadline

deadline_future_games_probabilities

In [0]:
# %sql
# -- Calculating each future game based on each team's current win percentage
# CREATE OR REPLACE TEMP VIEW deadline_future_game_probabilities AS

# SELECT
#     f.game_pk,
#     f.game_date,

#     f.away_team,
#     f.home_team,

#     aw.win_pct AS away_current_win_pct,
#     hw.win_pct AS home_current_win_pct,

#     -- Relative team strength
#     -- future Win probability per game based on current win pct
#     (
#         hw.win_pct /
#         (hw.win_pct + aw.win_pct)
#     ) AS home_current_game_probability,

#     (
#         aw.win_pct /
#         (hw.win_pct + aw.win_pct)
#     ) AS away_current_game_probability,

#     -- Caps the home win probability at 99%
#     ROUND(LEAST(
#         0.99,
#         (
#             home_current_win_pct /
#             (home_current_win_pct + away_current_win_pct)
#         ) + 0.03  -- home field advantage
#     ), 2) AS home_current_game_win_probability,

#     -- Floors the home win probability at 99%
#     ROUND(GREATEST(
#         0.01,
#         1 -
#         (
#             (
#                 home_current_win_pct /
#                 (home_current_win_pct + away_current_win_pct)
#             ) + 0.03
#         )
#     ), 2) AS away_current_game_win_probability



# FROM future_games f

# LEFT JOIN gold.mlb_deadline_standings aw
#     ON f.away_team = aw.team_name

# LEFT JOIN gold.mlb_deadline_standings hw
#     ON f.home_team = hw.team_name;


# SELECT
#     game_date,
#     away_team,
#     home_team,
#     away_current_game_win_probability AS away_probability,
#     home_current_game_win_probability AS home_probability
# FROM deadline_future_game_probabilities
# ORDER BY game_date, game_pk;

deadline_future_games_df

In [0]:
# # Same table as future_game_probabilites above but in Python
# deadline_future_games_df = spark.sql("""
#     SELECT
#         game_pk,
#         game_date,
#         away_team,
#         home_team,
#         away_current_game_win_probability AS away_probability,
#         home_current_game_win_probability AS home_probability
#     FROM deadline_future_game_probabilities
#     WHERE away_current_game_win_probability IS NOT NULL
#       AND home_current_game_win_probability IS NOT NULL
# """)

###Writing the future_games_df probabilites into Gold Layer

In [0]:
# deadline_future_games_df.write.mode("overwrite").saveAsTable(
#     "gold.mlb_deadline_game_probabilities"
# )

deadline_standings_df

In [0]:
# # Getting deadline standings table from gold layer
# deadline_standings_df = spark.sql("""
#     SELECT
#         team_name,
#         wins,
#         losses
#     FROM gold.mlb_deadline_standings
# """)

In [0]:
# deadline_games = deadline_future_games_df.collect()
# deadline_standings_rows = deadline_standings_df.collect()

# deadline_starting_records = {
#     row["team_name"]: {
#         "wins": row["wins"],
#         "losses": row["losses"]
#     }
#     for row in deadline_standings_rows
# }

# 10,000 Simulations of the future schedule after Trade Deadline (Aug 4th - Sept 27)

Only run once

In [0]:
# import random
# import copy

# deadline_num_simulations = 10000

# # Final standings
# deadline_all_simulations = []

# # Every individual game outcome
# deadline_simulation_game_results = []


# for sim in range(deadline_num_simulations):

#     # Start from original standings
#     deadline_sim_records = copy.deepcopy(deadline_starting_records)

#     # Simulate every remaining game
#     for game_number, game in enumerate(deadline_games, start=1):

#         away = game["away_team"]
#         home = game["home_team"]

#         away_prob = game["away_probability"]

#         # Generate random outcome between 0.0 and 1.0
#         if random.random() < away_prob:

#             winner = away
#             loser = home

#         else:

#             winner = home
#             loser = away


#         # Update records
#         deadline_sim_records[winner]["wins"] += 1
#         deadline_sim_records[loser]["losses"] += 1


#         # SAVE THE ACTUAL GAME RESULT
#         deadline_simulation_game_results.append({

#             "simulation": sim + 1,

#             "game_number": game_number,

#             "game_pk": game["game_pk"],

#             "game_date": game["game_date"],

#             "away_team": away,

#             "home_team": home,

#             "winner": winner,

#             "loser": loser,

#             "away_probability": away_prob

#         })


#     # Save final standings
#     for team, record in deadline_sim_records.items():

#         deadline_all_simulations.append({

#             "simulation": sim + 1,

#             "team": team,

#             "wins": record["wins"],

#             "losses": record["losses"]

#         })


# print(f"Completed {deadline_num_simulations:,} simulations")
# print(f"Final standings rows: {len(deadline_all_simulations):,}")
# print(f"Game result rows: {len(deadline_simulation_game_results):,}")

deadline_simulations_df (table)

12 teams have one or two missing games or an extra game played (most likely for makeups and doubleheaders)

In [0]:
# import pyspark.sql.functions as F

# # Each team's record for each simulation (10,000 sims for each team) in table form
# deadline_simulations_df = spark.createDataFrame(deadline_all_simulations)

# deadline_simulations_df = deadline_simulations_df.select(
#     F.col("simulation").alias("simulation"),
#     F.col("team").alias("team"),
#     "wins",
#     "losses"
# )
# display(deadline_simulations_df)

##Writing the deadline_simulations_df (all simulation results after the Trade Deadline) into Gold Layer

simulation_game_results (table) as deadline_simulations_df

In [0]:
# deadline_simulations_df = spark.createDataFrame(deadline_all_simulations)

# deadline_simulations_df.write.mode("overwrite").saveAsTable(
#     "gold.mlb_deadline_simulations"
# )

In [0]:
# # putting deadline_simulation_game_results into table form
# deadline_simulation_game_results_df = spark.createDataFrame(
#     deadline_simulation_game_results
# )

# deadline_simulation_game_results_df.createOrReplaceTempView(
#     "deadline_simulation_game_results"
# )

# display(deadline_simulation_game_results_df.limit(20))

deadline_final_wins_by_team

In [0]:
# from collections import defaultdict

# # Starting wins for every team
# deadline_starting_wins = {
#     team: record["wins"]
#     for team, record in deadline_starting_records.items()
# }


# # ==========================================
# # CALCULATE FINAL WINS FOR EVERY
# # TEAM IN EVERY SIMULATION
# # ==========================================

# deadline_final_wins_by_team = defaultdict(
#     lambda: defaultdict(int)
# )

# # Start every simulation with the team's current wins
# for team, wins in deadline_starting_wins.items():

#     for sim in range(1, deadline_num_simulations + 1):

#         deadline_final_wins_by_team[team][sim] = wins


# # Add simulated wins
# for row in deadline_simulation_game_results:

#     winner = row["winner"]
#     sim = row["simulation"]

#     if winner in deadline_final_wins_by_team:

#         deadline_final_wins_by_team[winner][sim] += 1


# print("Finished calculating final wins.")

selected_team_simulations (min, max, median with matched simulation)

In [0]:
# # ==========================================
# # FIND PESSIMISTIC / BASE / OPTIMISTIC
# # SIMULATION FOR EVERY TEAM
# # ==========================================

# deadline_selected_team_simulations = {}

# for team in deadline_starting_wins.keys():

#     results = sorted(
#         deadline_final_wins_by_team[team].items(),
#         key=lambda x: x[1]
#     )

#     # Minimum
#     min_simulation, min_wins = results[0]

#     # Median
#     median_index = len(results) // 2
#     median_simulation, median_wins = results[median_index]

#     # Maximum
#     max_simulation, max_wins = results[-1]

#     deadline_selected_team_simulations[team] = {

#         "Pessimistic": {
#             "simulation": min_simulation,
#             "final_wins": min_wins
#         },

#         "Base": {
#             "simulation": median_simulation,
#             "final_wins": median_wins
#         },

#         "Optimistic": {
#             "simulation": max_simulation,
#             "final_wins": max_wins
#         }
#     }

In [0]:
# for team in deadline_selected_team_simulations:

#     print(
#         f"{team}: "
#         f"Pessimistic = {deadline_selected_team_simulations[team]['Pessimistic']['final_wins']} "
#         f"(Sim {deadline_selected_team_simulations[team]['Pessimistic']['simulation']}), "
#         f"Base = {deadline_selected_team_simulations[team]['Base']['final_wins']} "
#         f"(Sim {deadline_selected_team_simulations[team]['Base']['simulation']}), "
#         f"Optimistic = {deadline_selected_team_simulations[team]['Optimistic']['final_wins']} "
#         f"(Sim {deadline_selected_team_simulations[team]['Optimistic']['simulation']})"
#     )

deadline_projection_paths

In [0]:
# # ==========================================
# # BUILD THREE SIMULATED PATHS Game-by-Game results FOR
# # EVERY TEAM
# # ==========================================

# deadline_projection_paths = []


# # Create quick lookup:
# # (team, simulation) -> scenario(s)

# deadline_selected_lookup = defaultdict(list)

# for team, scenarios in deadline_selected_team_simulations.items():

#     for scenario, info in scenarios.items():

#         deadline_selected_lookup[
#             (team, info["simulation"])
#         ].append(scenario)


# # Process every simulated game
# for row in deadline_simulation_game_results:

#     simulation = row["simulation"]

#     away = row["away_team"]
#     home = row["home_team"]

#     # Check both teams because either one
#     # could have one of its three selected simulations
#     for team in [away, home]:

#         key = (team, simulation)

#         if key not in deadline_selected_lookup:
#             continue

#         # Determine whether this is home or away
#         if team == away:
#             is_winner = row["winner"] == away
#         else:
#             is_winner = row["winner"] == home

#         for scenario in deadline_selected_lookup[key]:

#             deadline_projection_paths.append({
#                 "team": team,
#                 "scenario": scenario,
#                 "simulation": simulation,
#                 "game_pk": row["game_pk"],
#                 "game_date": row["game_date"],
#                 "game_number": None,
#                 "away_team": away,
#                 "home_team": home,
#                 "winner": row["winner"],
#                 "won_game": 1 if is_winner else 0
#             })


# print(
#     f"Created {len(deadline_projection_paths):,} projection rows."
# )

Calculating the projected Wins after the Trade Deadline (Aug 4th - Sept 27)

In [0]:
# # ==========================================
# # CALCULATE CUMULATIVE PROJECTED WINS
# # ==========================================

# # Sort chronologically
# deadline_projection_paths = sorted(
#     deadline_projection_paths,
#     key=lambda x: (
#         x["team"],
#         x["scenario"],
#         x["game_date"],
#         x["game_pk"]
#     )
# )


# current_team = None
# current_scenario = None
# current_wins = None
# game_number = 0


# for row in deadline_projection_paths:

#     # New team/scenario combination
#     if (
#         row["team"] != current_team
#         or row["scenario"] != current_scenario
#     ):

#         current_team = row["team"]
#         current_scenario = row["scenario"]

#         current_wins = deadline_starting_wins[current_team]

#         game_number = 0

#     game_number += 1

#     current_wins += row["won_game"]

#     row["game_number"] = game_number
#     row["projected_wins"] = current_wins

deadline_all_team_projection_df (table)

In [0]:
# deadline_all_team_projection_df = spark.createDataFrame(
#     deadline_projection_paths
# )

# display(
#     deadline_all_team_projection_df
#     .orderBy(
#         "team",
#         "game_date",
#         "scenario"
#     )
# )

###Writing deadline_all_team_projected_df into Gold Layer

In [0]:
# deadline_all_team_projection_df.write \
#     .mode("overwrite") \
#     .saveAsTable(
#         "gold.mlb_deadline_team_simulation_projection"
#     )

## Checking by looking at one team (Milwaukee Brewers)

In [0]:
# brewers_check = [
#     x for x in deadline_projection_paths
#     if x["team"] == "Milwaukee Brewers"
# ]

# for scenario in ["Pessimistic", "Base", "Optimistic"]:

#     results = [
#         x for x in brewers_check
#         if x["scenario"] == scenario
#     ]

#     print(
#         scenario,
#         "→",
#         results[-1]["projected_wins"],
#         "wins",
#         "| Simulation:",
#         results[0]["simulation"]
#     )

In [0]:
# %sql
# SELECT 
#     * 
# FROM gold.mlb_deadline_team_simulation_projection
# WHERE team = 'Milwaukee Brewers'

#Calculating Cumulative ACTUAL record from Trade Deadline to Current (Aug 4th - Current)

In [0]:
%sql
-- Total runs per team
CREATE OR REPLACE TEMP VIEW team_game_runs AS
SELECT
    game_pk,
    team_id,
    SUM(runs) AS runs_scored
FROM gold.fact_batting_stats
GROUP BY game_pk, team_id;


-- Total runs scored per game
CREATE OR REPLACE TEMP VIEW game_results AS
SELECT 
    g.game_pk,
    g.home_team_id,
    h.runs_scored AS home_runs,
    g.away_team_id,
    a.runs_scored AS away_runs
FROM gold.dim_games AS g
LEFT JOIN team_game_runs AS h ON g.game_pk = h.game_pk AND g.home_team_id = h.team_id
LEFT JOIN team_game_runs AS a ON g.game_pk = a.game_pk AND g.away_team_id = a.team_id
WHERE g.game_status IN ('Final', 'Pre-Game', 'Scheduled', 'Completed Early');


-- wins and losses per team per game
CREATE OR REPLACE TEMP VIEW team_outcomes AS
SELECT
    game_pk,
    home_team_id AS team_id,
    CASE WHEN home_runs > away_runs THEN 1 ELSE 0 END AS Win,
    CASE WHEN home_runs < away_runs THEN 1 ELSE 0 END AS Loss,
    home_runs - away_runs AS run_diff
FROM game_results
UNION ALL
SELECT
    game_pk,
    away_team_id AS team_id,
    CASE WHEN away_runs > home_runs THEN 1 ELSE 0 END AS Win,
    CASE WHEN away_runs < home_runs THEN 1 ELSE 0 END AS Loss,
    away_runs - home_runs AS run_diff
FROM game_results;

In [0]:
# ==========================================
# BUILD ACTUAL WIN PATH
# Game-by-game results for EVERY TEAM
# ==========================================

from collections import defaultdict

actual_win_path = []

# Starting wins as of August 3
starting_wins = {
    row["team_name"]: row["wins"]
    for row in spark.sql("""
        SELECT
            team_name,
            wins
        FROM gold.mlb_deadline_standings
    """).collect()
}


# Get actual completed games from August 4 onward with winners
actual_games = spark.sql("""
    SELECT
        g.game_pk,
        g.game_date,
        away_t.team_name AS away_team,
        home_t.team_name AS home_team,
        CASE 
            WHEN SUM(CASE WHEN o.team_id = g.away_team_id THEN o.Win ELSE 0 END) = 1 THEN away_t.team_name
            WHEN SUM(CASE WHEN o.team_id = g.home_team_id THEN o.Win ELSE 0 END) = 1 THEN home_t.team_name
            ELSE NULL
        END AS winner
    FROM gold.dim_games g
    JOIN team_outcomes o ON g.game_pk = o.game_pk
    JOIN gold.dim_teams away_t ON g.away_team_id = away_t.team_id
    JOIN gold.dim_teams home_t ON g.home_team_id = home_t.team_id
    WHERE g.game_date >= '2026-08-04'
      AND g.game_status IN ('Final', 'Completed Early', 'Scheduled', 'Pre-Game')
    GROUP BY g.game_pk, g.game_date, away_t.team_name, home_t.team_name
    ORDER BY g.game_date, g.game_pk
""").collect()


# Track cumulative wins for each team
actual_cumulative_wins = defaultdict(int)

# Process every actual game
for row in actual_games:

    away = row["away_team"]
    home = row["home_team"]
    winner = row["winner"]

    if winner is None:
        continue

    # Both teams play this game
    for team in [away, home]:

        won_game = 1 if team == winner else 0

        actual_cumulative_wins[team] += won_game

        actual_win_path.append({
            "team_name": team,
            "game_date": row["game_date"],
            "game_pk": row["game_pk"],
            "away_team": away,
            "home_team": home,
            "winner": winner,
            "won_game": won_game,
            "actual_wins": starting_wins[team] + actual_cumulative_wins[team]
        })


print(
    f"Created {len(actual_win_path):,} actual win-path rows."
)

In [0]:
actual_win_path_df = spark.createDataFrame(
    actual_win_path
)

display(
    actual_win_path_df
    .orderBy(
        "team_name",
        "game_date",
        "game_pk",
        "away_team",
        "home_team",
        "winner",
        "won_game",
        "actual_wins"
    )
)

In [0]:
actual_win_path_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "gold.mlb_actual_win_path"
    )

###Checking actual cumulative wins with Milwaukee Brewers"

In [0]:
# %sql
# SELECT *
# FROM gold.mlb_actual_win_path
# WHERE team_name = 'Milwaukee Brewers'
# ORDER BY game_date, game_pk;